# 02C 数据清洗与特征构造

> 🟢 **Level A · 必须掌握** | 完成标准：`raw table → audit → clean → construct features → select columns → ML-ready table`。

本节开始使用公开 COFSpace 的真实 CoRE-COF CO₂ 1 bar 数据。


In [ ]:
import pandas as pd
url='https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv'
df=pd.read_csv(url)
display(df.head())
display(pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'missing_%':(100*df.isna().mean()).round(2),'unique':df.nunique()}))
print('duplicates =',df.duplicated().sum())


## Feature / target / provenance
`CO2-1 bar (mol/kg)` 是 target。PLD、LCD、surface area、porosity 和元素比例是候选 features；ID、DOI、filename 应保留用于追踪，但通常不是模型输入。


In [ ]:
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
work=df[features+[target]].copy()
work['LCD_PLD_ratio']=work['LCD (Å)']/work['PLD (Å)']
work['heteroatom_%']=work[['%N','%O','%Metalloid','%Halogen','%Ametal']].sum(axis=1)
display(work.head())


## Missing、scaling 与 selection
Missing 不等于 0。Scaling 对 KNN、linear/logistic models 常重要，对树模型通常不是必须。先删除 target-derived、ID、不可获得或重复 feature，再考虑复杂选择方法。

### 02C 完成标准
能够产生明确的 `X`、`y`，并记录 feature 的来源、单位与处理方式。


## 把审计落实为清洗决策
保留源表行号用于追溯；行号不等于已验证的材料 ID。缺失 target 不应填补。显式转换数值、记录剔除行；PLD 非正时不计算比值。中位数填补和 scaling 要放进训练折内的 Pipeline（03B）。


In [ ]:
import numpy as np
numeric = df[features + [target]].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
rejected = df.loc[numeric[target].isna()].copy()
clean = numeric.loc[numeric[target].notna()].copy()
clean['LCD_PLD_ratio'] = clean['LCD (Å)'] / clean['PLD (Å)'].where(clean['PLD (Å)'] > 0)
clean['heteroatom_%'] = clean[['%N','%O','%Metalloid','%Halogen','%Ametal']].sum(axis=1, min_count=5)
X = clean.drop(columns=target)
y = clean[target]
print('kept / missing target:', len(clean), len(rejected))
print('duplicate feature rows (investigate; do not silently drop):', X.duplicated().sum())
display(X.isna().sum().to_frame('missing'))


## 练习与交付
提交特征字典（名称、单位、来源、预测时是否可获得）、剔除记录和 X/y。解释为什么 missing 不等于零、重复特征行不一定是重复材料。03A 使用这类表训练；03B 负责验证预处理是否泄漏。


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
